# Hangman RNN: evaluation on the self-hosted engine

`hangman.engine.HangmanEngine` plays the role of a game server: it draws secret words from `data/words_test.txt` (disjoint from the training dictionary), applies the rules (6 wrong guesses, repeated letter ends the game) and only ever shows the player the masked word.

Run from the `notebooks/` directory.

In [1]:
import sys
sys.path.insert(0, "..")

from hangman import HangmanBasic, HangmanRNN, load_model
from hangman.engine import HangmanEngine
from hangman.evaluate import evaluate, play_game

## One game, step by step

In [2]:
rnn = HangmanRNN(timeout=None, model=load_model())
play_game(rnn, HangmanEngine(seed=7), recorded=False, verbose=True);

Game 44fd8df0670d | tries: 6 | word: _ _ _ _ _ _ _ _ _ _ _ _
  guess e | tries: 6 | word: _ _ _ _ _ _ _ _ _ _ _ e
  guess i | tries: 6 | word: _ _ _ _ _ _ i _ _ _ _ e
  guess a | tries: 6 | word: _ a _ _ _ _ i _ _ a _ e
  guess t | tries: 5 | word: _ a _ _ _ _ i _ _ a _ e
  guess n | tries: 4 | word: _ a _ _ _ _ i _ _ a _ e
  guess o | tries: 4 | word: _ a _ _ o _ i _ _ a _ e
  guess r | tries: 4 | word: _ a r _ o _ i _ _ a _ e
  guess l | tries: 4 | word: _ a r _ o _ i l l a _ e
  guess g | tries: 4 | word: _ a r _ o _ i l l a g e
  guess c | tries: 3 | word: _ a r _ o _ i l l a g e
  guess b | tries: 3 | word: b a r b o _ i l l a g e
  guess x | tries: 2 | word: b a r b o _ i l l a g e
  guess d | tries: 1 | word: b a r b o _ i l l a g e
  guess p | tries: 0 | word: b a r b o _ i l l a g e
  -> failed (# of tries exceeded.) answer: barbouillage



## Benchmark: 1,000 games each, same words for both strategies (seed 0)

In [3]:
import pandas as pd

results = {
    "Dictionary baseline": evaluate(HangmanBasic(), games=1000, seed=0, progress=False),
    "Bidirectional GRU": evaluate(rnn, games=1000, seed=0, progress=False),
}
pd.DataFrame(results).T[["games", "wins", "success_rate"]].astype({"games": int, "wins": int})

,games,wins,success_rate
Dictionary baseline,1000,188,0.188
Bidirectional GRU,1000,527,0.527


## Playing over HTTP

The same engine is exposed as a REST API in `hangman/server.py`:

```bash
pip install fastapi uvicorn
uvicorn hangman.server:app --port 8000   # docs at http://localhost:8000/docs
```

```python
import requests
g = requests.post("http://localhost:8000/games", json={}).json()
g = requests.post(f"http://localhost:8000/games/{g['game_id']}/guess", json={"letter": "e"}).json()
```